## Streaming rides_raw table transformation

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
rides_schema = StructType([StructField('ride_id', StringType(), True), StructField('confirmation_number', StringType(), True), StructField('passenger_id', StringType(), True), StructField('driver_id', StringType(), True), StructField('vehicle_id', StringType(), True), StructField('pickup_location_id', StringType(), True), StructField('dropoff_location_id', StringType(), True), StructField('vehicle_type_id', LongType(), True), StructField('vehicle_make_id', LongType(), True), StructField('payment_method_id', LongType(), True), StructField('ride_status_id', LongType(), True), StructField('pickup_city_id', LongType(), True), StructField('dropoff_city_id', LongType(), True), StructField('cancellation_reason_id', LongType(), True), StructField('passenger_name', StringType(), True), StructField('passenger_email', StringType(), True), StructField('passenger_phone', StringType(), True), StructField('driver_name', StringType(), True), StructField('driver_rating', DoubleType(), True), StructField('driver_phone', StringType(), True), StructField('driver_license', StringType(), True), StructField('vehicle_model', StringType(), True), StructField('vehicle_color', StringType(), True), StructField('license_plate', StringType(), True), StructField('pickup_address', StringType(), True), StructField('pickup_latitude', DoubleType(), True), StructField('pickup_longitude', DoubleType(), True), StructField('dropoff_address', StringType(), True), StructField('dropoff_latitude', DoubleType(), True), StructField('dropoff_longitude', DoubleType(), True), StructField('distance_miles', DoubleType(), True), StructField('duration_minutes', LongType(), True), StructField('booking_timestamp', TimestampType(), True), StructField('pickup_timestamp', StringType(), True), StructField('dropoff_timestamp', StringType(), True), StructField('base_fare', DoubleType(), True), StructField('distance_fare', DoubleType(), True), StructField('time_fare', DoubleType(), True), StructField('surge_multiplier', DoubleType(), True), StructField('subtotal', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('total_fare', DoubleType(), True), StructField('rating', DoubleType(), True)])

In [0]:
df = spark.read.table("uber.bronze.rides_raw")
df_parsed = df.withColumn("parsed_rides", from_json(col("rides"),rides_schema))\
    .select("parsed_rides.*")

display(df_parsed)

In [0]:
# # fetch the schema from bulk rides table. beacuse we want to transform streaming table schema like bulk ride table

# df = spark.sql("select * from uber.bronze.bulk_rides")
# df.schema

## Jinja template for OBT

In [0]:
pip install jinja2

In [0]:
dbutils.library.restartPython()

In [0]:
%sql
select 
     staging_rides.*
from    
    uber.bronze.staging_rides staging_rides
left join
    uber.bronze.map_vehicle_types map_vehicle_types
on
    staging_rides.vehicle_type_id = map_vehicle_types.vehicle_type_id
left join
    uber.bronze.map_vehicle_makes map_vehicle_makes
on
    staging_rides.vehicle_make_id = map_vehicle_makes.vehicle_make_id



In [0]:
jinja_config = [
    {
        "table" : "uber.bronze.staging_rides staging_rides",
        "select" : 'staging_rides.ride_id, staging_rides.confirmation_number, staging_rides.passenger_id, staging_rides.driver_id, staging_rides.vehicle_id, staging_rides.pickup_location_id, staging_rides.dropoff_location_id, staging_rides.vehicle_type_id, staging_rides.vehicle_make_id, staging_rides.payment_method_id, staging_rides.ride_status_id, staging_rides.pickup_city_id, staging_rides.dropoff_city_id, staging_rides.cancellation_reason_id, staging_rides.passenger_name, staging_rides.passenger_email, staging_rides.passenger_phone, staging_rides.driver_name, staging_rides.driver_rating, staging_rides.driver_phone, staging_rides.driver_license, staging_rides.vehicle_model, staging_rides.vehicle_color, staging_rides.license_plate, staging_rides.pickup_address, staging_rides.pickup_latitude, staging_rides.pickup_longitude, staging_rides.dropoff_address, staging_rides.dropoff_latitude, staging_rides.dropoff_longitude, staging_rides.distance_miles, staging_rides.duration_minutes, staging_rides.booking_timestamp, staging_rides.pickup_timestamp, staging_rides.dropoff_timestamp, staging_rides.base_fare, staging_rides.distance_fare, staging_rides.time_fare, staging_rides.surge_multiplier, staging_rides.subtotal, staging_rides.tip_amount, staging_rides.total_fare, staging_rides.rating',
        "where" : ""
    },
    {
        "table" : "uber.bronze.map_vehicle_makes map_vehicle_makes",
        "select" : "map_vehicle_makes.vehicle_make",
        "where" : "",
        "on" : "staging_rides.vehicle_make_id = map_vehicle_makes.vehicle_make_id"
    },
    {
        "table" : "uber.bronze.map_vehicle_types map_vehicle_types",
        "select" : "map_vehicle_types.vehicle_type,map_vehicle_types.description,map_vehicle_types.base_rate,map_vehicle_types.per_mile,map_vehicle_types.per_minute",
        "where" : "",
        "on" : "staging_rides.vehicle_type_id = map_vehicle_types.vehicle_type_id"
    },
    {
        "table" : "uber.bronze.map_ride_statuses map_ride_statuses",
        "select" : "map_ride_statuses.ride_status",
        "where" : "",
        "on" : "staging_rides.ride_status_id = map_ride_statuses.ride_status_id"
    },
    {
        "table" : "uber.bronze.map_payment_methods map_payment_methods",
        "select" : "map_payment_methods.payment_method, map_payment_methods.is_card, map_payment_methods.requires_auth",
        "where" : "",
        "on" : "staging_rides.payment_method_id = map_payment_methods.payment_method_id"
    },
    {
        "table" : "uber.bronze.map_cities map_cities",
        "select" : "map_cities.city as pickup_city, map_cities.state, map_cities.region, map_cities.updated_at as city_updated_at",
        "where" : "",
        "on" : "staging_rides.pickup_city_id = map_cities.city_id"
    },
    {
        "table" : "uber.bronze.map_cancellation_reasons map_cancellation_reasons",
        "select" : "map_cancellation_reasons.cancellation_reason",
        "where" : "",
        "on" : "staging_rides.cancellation_reason_id = map_cancellation_reasons.cancellation_reason_id"
    }
]

In [0]:
from jinja2 import Template

jinja_str = """
SELECT
    {% for config in jinja_config %}
        {{ config.select }}
            {% if not loop.last %}
                ,
            {% endif %}
    {% endfor %}
FROM 
    {% for config in jinja_config %}
        {% if loop.first %}
            {{ config.table }}
        {% else %}
            LEFT JOIN {{config.table}} ON {{ config.on }}
        {% endif %}
    {% endfor %}

    {% for config in jinja_config %}
        {% if loop.first %}
            {% if config.where != "" %} 
            WHERE
            {% endif %}
        {% endif %}

        {{ config.where }}
        {% if not loop.last %}
            {%if config.where != "" %}
            AND
            {% endif %}
        {% endif %}
    {% endfor %}
"""


template = Template(jinja_str)
rendered_query = template.render(jinja_config=jinja_config)



In [0]:
print(rendered_query)

In [0]:
display(spark.sql(rendered_query))

In [0]:
# from jinja2 import Template

# jinja_str = """
# SELECT
#     {{ jinja_config[0].select }},
#     {{ jinja_config[1].select }},
#     {{ jinja_config[2].select }}
# FROM {{ jinja_config[0].table }}
# LEFT JOIN {{ jinja_config[2].table }} ON {{ jinja_config[2].on }}
# LEFT JOIN {{ jinja_config[1].table }} ON {{ jinja_config[1].on }}
# """

# template = Template(jinja_str)
# rendered_query = template.render(jinja_config=jinja_config)

In [0]:
%sql
select current_timestamp()